In [3]:
!pip install VNSFintech

   ---------------------------------------- 0.0/41.9 kB ? eta -:--:--
   --------- ------------------------------ 10.2/41.9 kB ? eta -:--:--
   ---------------------------------------  41.0/41.9 kB 495.5 kB/s eta 0:00:01
   ---------------------------------------- 41.9/41.9 kB 405.8 kB/s eta 0:00:00


In [5]:
%pip install ta

  Using cached ta-0.11.0.tar.gz (25 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29422 sha256=80074657c63af454831b18539b7e2ed63060313fab7e8cd9785b9664ae472c1e
  Stored in directory: c:\users\qui\appdata\local\pip\cache\wheels\a1\d7\29\7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta
Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install prophet

  Using cached cmdstanpy-1.2.5-py3-none-any.whl.metadata (4.0 kB)
     ---------------------------------------- 0.0/49.7 kB ? eta -:--:--
     ------------------------ --------------- 30.7/49.7 kB 1.3 MB/s eta 0:00:01
     ------------------------------- ------ 41.0/49.7 kB 487.6 kB/s eta 0:00:01
     -------------------------------------- 49.7/49.7 kB 420.2 kB/s eta 0:00:00
  Using cached stanio-0.5.1-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/13.3 MB ? eta -:--:--
   ---------------------------------------- 0.1/13.3 MB ? eta -:--:--
    --------------------------------------- 0.2/13.3 MB 2.2 MB/s eta 0:00:06
   - -------------------------------------- 0.3/13.3 MB 2.7 MB/s eta 0:00:05
   -- ------------------------------------- 0.7/13.3 MB 4.1 MB/s eta 0:00:04
   -- ------------------------------------- 0.9/13.3 MB 3.9 MB/s eta 0:00:04
   --- ------------------------------------ 1.2/13.3 MB 4.3 MB/s eta 0:00:03
   ---- --------------------------

In [12]:
# 1. Import thư viện
from VNSFintech import *
import pandas as pd
import ta
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [13]:
# 2. Load dữ liệu cổ phiếu
df = history(symbol='KBC', start='2020-01-01', end=pd.Timestamp.today().strftime('%Y-%m-%d'), time='days')
df.rename(columns={'time': 'ds', 'close': 'y'}, inplace=True)
df['ds'] = pd.to_datetime(df['ds'])
df.sort_values('ds', inplace=True)

In [14]:
df.head()

,symbol,ds,open,y,high,low,volume
1440,KBC,2020-01-02,11.48,11.40,11.59,11.32,3144560
1439,KBC,2020-01-03,11.40,11.48,11.51,11.40,1330450
1438,KBC,2020-01-06,11.40,11.32,11.51,11.29,2068260
1437,KBC,2020-01-07,11.32,11.21,11.44,11.21,1505630
1436,KBC,2020-01-08,11.18,11.40,11.48,11.02,3810860


In [15]:
# 3. Tính chỉ báo kỹ thuật
df['ema20'] = ta.trend.EMAIndicator(df['y'], 20).ema_indicator()
macd = ta.trend.MACD(df['y'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['adx'] = ta.trend.ADXIndicator(df['high'], df['low'], df['y']).adx()
df['psar'] = ta.trend.PSARIndicator(df['high'], df['low'], df['y']).psar()

df['rsi'] = ta.momentum.RSIIndicator(df['y']).rsi()
df['roc'] = ta.momentum.ROCIndicator(df['y']).roc()
df['cci'] = ta.trend.CCIIndicator(df['high'], df['low'], df['y']).cci()
df['williams_r'] = ta.momentum.WilliamsRIndicator(df['high'], df['low'], df['y']).williams_r()
df['stoch'] = ta.momentum.StochasticOscillator(df['high'], df['low'], df['y']).stoch()


df['obv'] = ta.volume.OnBalanceVolumeIndicator(df['y'], df['volume']).on_balance_volume()
df['mfi'] = ta.volume.MFIIndicator(df['high'], df['low'], df['y'], df['volume']).money_flow_index()
df['cmf'] = ta.volume.ChaikinMoneyFlowIndicator(df['high'], df['low'], df['y'], df['volume']).chaikin_money_flow()

df['atr'] = ta.volatility.AverageTrueRange(df['high'], df['low'], df['y']).average_true_range()
bb = ta.volatility.BollingerBands(df['y'])
df['bollinger_h'] = bb.bollinger_hband()
df['bollinger_l'] = bb.bollinger_lband()
dc = ta.volatility.DonchianChannel(df['high'], df['low'], df['y'])
df['donchian_h'] = dc.donchian_channel_hband()
df['donchian_l'] = dc.donchian_channel_lband()


In [16]:
# 4. Loại bỏ các dòng NaN do tính chỉ báo
df.dropna(inplace=True)

In [17]:
from sklearn.preprocessing import StandardScaler

# Chọn các cột chỉ báo để scale
features = ['ema20','psar','rsi','obv','bollinger_h','bollinger_l','atr','macd','macd_signal']

# Scale các chỉ báo
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

In [18]:
# 5. Chia tập train/test theo tỷ lệ 80:20
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [19]:
from sklearn.preprocessing import MinMaxScaler

def prepare_scaled_data(df, top_features):
    # Chỉ scale các chỉ báo ngoại sinh
    scaler = MinMaxScaler()
    scaled_features = scaler.fit_transform(df[top_features])
    scaled_df = df[['y']].copy()
    for i, col in enumerate(top_features):
        scaled_df[col] = scaled_features[:, i]
    
    scaled_df['ds'] = df.index
    scaled_df = scaled_df[['ds', 'y'] + top_features]
    return scaled_df, scaler


In [20]:
from prophet.diagnostics import cross_validation, performance_metrics
from prophet import Prophet
from itertools import product
import random

# Define search space
param_grid = {
    'changepoint_prior_scale': [0.001, 0.01, 0.05, 0.1, 0.5],
    'seasonality_prior_scale': [1.0, 5.0, 10.0, 20.0],
    'seasonality_mode': ['additive', 'multiplicative']
}

# Create a list of all possible combinations
all_params = [dict(zip(param_grid, v)) for v in product(*param_grid.values())]
random.shuffle(all_params)  # Optional: randomize to speed up

features = ['rsi', 'macd', 'macd_signal', 'ema20', 'roc', 'williams_r', 'cci']
best_params = None
best_rmse = float('inf')

print("🔍 Tuning hyperparameters...")

for params in all_params[:15]:  # kiểm tra 15 cấu hình đầu tiên
    model = Prophet(
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        seasonality_mode=params['seasonality_mode'],
        daily_seasonality=False
    )
    
    for reg in features:
        model.add_regressor(reg)

    model.fit(train_df[['ds', 'y'] + features])

    # Cross-validation trên tập train
    df_cv = cross_validation(model, horizon='30 days', initial='365 days', period='90 days', parallel="processes")
    df_p = performance_metrics(df_cv, rolling_window=1)

    rmse = df_p['rmse'].values[0]

    print(f"Params: {params}, RMSE: {rmse:.2f}")
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params

print("🎯 Best Params Found:", best_params)


🔍 Tuning hyperparameters...


11:44:19 - cmdstanpy - INFO - Chain [1] start processing
11:44:19 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.


Params: {'changepoint_prior_scale': 0.01, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative'}, RMSE: 0.96


11:44:28 - cmdstanpy - INFO - Chain [1] start processing
11:44:29 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.


Params: {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 20.0, 'seasonality_mode': 'multiplicative'}, RMSE: 2.55


...

Params: {'changepoint_prior_scale': 0.01, 'seasonality_prior_scale': 1.0, 'seasonality_mode': 'multiplicative'}, RMSE: 0.95


11:46:35 - cmdstanpy - INFO - Chain [1] start processing
11:46:36 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.


Params: {'changepoint_prior_scale': 0.5, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive'}, RMSE: 2.12


11:46:50 - cmdstanpy - INFO - Chain [1] start processing
11:46:51 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.


Params: {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 1.0, 'seasonality_mode': 'multiplicative'}, RMSE: 2.96
🎯 Best Params Found: {'changepoint_prior_scale': 0.01, 'seasonality_prior_scale': 20.0, 'seasonality_mode': 'additive'}


In [22]:
from prophet import Prophet

model = Prophet(
    changepoint_prior_scale=best_params['changepoint_prior_scale'],
    seasonality_prior_scale=best_params['seasonality_prior_scale'],
    seasonality_mode=best_params['seasonality_mode'],
    daily_seasonality=False
)


In [23]:
for reg in features:
    model.add_regressor(reg)

In [24]:
# Huấn luyện mô hình
model.fit(train_df[['ds', 'y'] + features])

11:47:41 - cmdstanpy - INFO - Chain [1] start processing
11:47:41 - cmdstanpy - INFO - Chain [1] done processing


In [25]:
# Dự báo trên tập test
future = test_df[['ds'] + features]
forecast = model.predict(future)

In [26]:
# Gộp kết quả để đánh giá
result = test_df[['ds', 'y']].copy()
result['yhat'] = forecast['yhat'].values

In [27]:
# Đánh giá mô hình
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(result['y'], result['yhat'])
rmse = np.sqrt(mean_squared_error(result['y'], result['yhat']))
print(f"✅ MAE: {mae:.2f}, RMSE: {rmse:.2f}")

✅ MAE: 0.56, RMSE: 0.68
